In [ ]:
%pip install openai --upgrade

In [1]:
import getpass
import os

# Set up your OpenAI client
if not os.getenv("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass.getpass("Enter your OpenAI API key: ")

MODEL = "gpt-5.4-mini"

In [2]:
from openai import OpenAI
client = OpenAI()

def run_llm(user_prompt: str, model: str = MODEL, system_prompt: str = None):
    response = client.responses.create(
        model=model,
        instructions=system_prompt,
        input=user_prompt
    )
    return response.output_text

run_llm('what is the meaning of life?')

'There isn’t one universally agreed “meaning of life,” but there are a few common ways people answer it:\n\n- **Biological:** to survive, reproduce, and continue life.\n- **Philosophical:** to create your own meaning through choices, values, and relationships.\n- **Relational:** to love, be loved, and care for others.\n- **Experiential:** to learn, grow, and experience as much of life as you can.\n- **Spiritual/religious:** to serve a higher purpose or follow a divine path.\n\nA concise answer many people settle on is: **the meaning of life is whatever you choose to build it around**—often connection, purpose, growth, and kindness.\n\nIf you want, I can also give you:\n1. a philosophical answer,  \n2. a scientific answer, or  \n3. a funny answer.'

In [3]:
import re

task = """
Write a one-sentence bedtime story about a unicorn, for a five year old girl
"""

GENERATOR_PROMPT = """
Your goal is to complete the task based on <user input>. If there are feedback
from your previous generations, you should reflect on them to improve your solution

Output your answer concisely in the following format:

Thoughts:
[Your understanding of the task and feedback and how you plan to improve]

Response:
[Your response here]
"""

def generate(task: str, generator_prompt: str, context: str="") -> tuple[str, str]:
    """Generate and improve a solution based on feedback."""
    full_prompt = f"{generator_prompt}\n{context}\nTask: {task}" if context else f"{generator_prompt}\nTask: {task}"

    response = run_llm(full_prompt)

    print("\n## Generation start")
    print(f"Output:\n{response}\n")

    return response


EVALUATOR_PROMPT = """
Evaluate this following response for:
1. age appropriateness
2. is only ten words or fewer
3. style and best practices

You should be evaluating only and not attempting to solve the task.

Only output "PASS" if all criteria are met and you have no further suggestions for improvements.

Provide detailed feedback if there are areas that need improvement. You should specify what needs improvement and why.

Return in this format:
Status: [PASS/FAIL]
Feedback: [Your feedback here]
"""

def evaluate(task: str, evaluator_prompt: str, generated_content: str) -> tuple[str, str]:
    """Evaluate if a solution meets requirements"""

    full_prompt = f"{evaluator_prompt}\nOriginal task: {task}\nContent to evaluate: {generated_content}"

    response = run_llm(full_prompt)

    status_match = re.search(r"Status:\s*(.*?)(?:\n|$)", response, re.IGNORECASE)
    feedback_match = re.search(r"Feedback:\s*([\s\S]*)", response, re.IGNORECASE)

    if status_match is None or feedback_match is None:
        raise ValueError("Could not parse evaluation response. Expected format: 'Status: [PASS/FAIL]\\nFeedback: [feedback]'")

    evaluation = status_match.group(1).strip()
    feedback = feedback_match.group(1).strip()

    print("## Evaluation start")
    print(f"Status: {evaluation}")
    print(f"Feedback: {feedback}")

    return evaluation, feedback

def loop_workflow(task: str, generator_prompt: str, evaluator_prompt: str, context: str = "") -> tuple[str, list[dict]]:
    """Keep generating and evaluating until the evaluator passes the last generated response."""
    memory = []

    response = generate(task, generator_prompt)
    memory.append(response)

    max_iterations = 5
    while max_iterations > 0:
        evaluation, feedback = evaluate(task, evaluator_prompt, response)

        if evaluation.upper() == "PASS":
            return response

        context = "\n".join([
            "Previous attempts:",
            *[f"- {m}" for m in memory],
            f"\nFeedback: {feedback}"
        ])
        response = generate(task, generator_prompt, context)
        memory.append(response)

        max_iterations -= 1

loop_workflow(task, GENERATOR_PROMPT, EVALUATOR_PROMPT)


## Generation start
Output:
Thoughts:
I understand the task is to write a single, simple, gentle bedtime story sentence about a unicorn that is sweet and suitable for a five-year-old girl.

Response:
As the moon twinkled softly in the sky, a kind little unicorn with a sparkling pink mane tiptoed through the sleepy garden, leaving tiny rainbow dreams for the stars to tuck into bed.

## Evaluation start
Status: FAIL
Feedback: The response is age-appropriate and child-friendly, but it does **not** meet the “ten words or fewer” requirement. It is a full sentence, but far longer than 10 words. For best practices, it should be much shorter and simpler to match the instruction exactly.

## Generation start
Output:
Thoughts:
I need to keep it as one sentence, sweet, bedtime-themed, unicorn-focused, and within ten words or fewer to address the feedback.

Response:
The unicorn slept softly under a twinkling moon.

## Evaluation start
Status: PASS
Feedback: 


'Thoughts:\nI need to keep it as one sentence, sweet, bedtime-themed, unicorn-focused, and within ten words or fewer to address the feedback.\n\nResponse:\nThe unicorn slept softly under a twinkling moon.'